# One locked GPU XGBoost diagnostic — clean Task2

This is a **joint-training tabular baseline**, not a continual-learning method, Transformer replacement, or poisoning defense. Historical test results were already inspected; the final test pass is exploratory. The two completed 12-epoch Transformer schedules both failed validation eligibility: A accuracy/macro-F1/Infiltration recall 95.43/69.61/23.58%; B 96.89/69.65/16.47%. Neither opened test.

Attach (1) the complete fixed prepared-array folder, (2) the verified Task1 run folder, and (3) the original discrimination `results.zip` or its complete extracted `task2_discrimination_study` folder. Keep them private. Enable Kaggle GPU and Internet. This notebook does not install or upgrade XGBoost; it checks the image's installed build and runs a one-tree CUDA compatibility probe before touching research data. A CPU fallback is an error.

**Predeclared protocol:** exactly 100,000 Benign + 10,000 each old attack replay rows from the saved manifest, plus all 70,076 new-class training rows; 54 features with the frozen Task1 continual transform. One XGBoost multiclass GPU-hist configuration: seed 42, max depth 6, eta 0.05, subsample/column sample 0.8, max bin 256, up to 500 rounds, weighted validation-logloss early stopping after 30 stale rounds. Train-only class weights are `min(5, sqrt(max_train_class_count / class_count))`, normalized to mean one; the same fixed mapping weights validation logloss only. Full validation/test metrics are unweighted. The best validation round and model are saved and locked before one test evaluation. Report every class; in particular never hide Infiltration behind average-new recall.

Methodological context: [tabular-benchmark](https://github.com/LeoGrin/tabular-benchmark) motivates a strong tabular tree comparator; [WTMC2021 CICIDS2017 troubleshooting](https://intrusion-detection.distrinet-research.be/WTMC2021/) motivates caution about flow labels and features. Neither source validates this combined collection or its labels. No benchmark code or relabelling is imported.

Publication note: this notebook pins the verified base commit and **exact runner/test file hashes**. The runner/test files must be published to `feature/unified-novelty` before Kaggle execution. Hash verification fails closed if publication is incomplete or files differ.

In [ ]:
from pathlib import Path
from hashlib import sha256
import subprocess, sys
BASE_SHA = '495ea48ff428f2895abb91c110ff353ef5fcb1e5'
PINNED_FILES = {
    'scripts/run_task2_xgboost_baseline.py': 'ef1259c4b6e053faa23328e30d7731d80587af874e01b045fe5cccb48d1f44cd',
    'tests/test_task2_xgboost_baseline.py': '3dabb0cee04e017e420ba0d38375d16ce5f98d3e746b104a915dec7d8c912687',
}
REPO = Path('/kaggle/working/poison-resilient-continual-nids')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', 'feature/unified-novelty:refs/remotes/origin/feature/unified-novelty'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '--detach', BASE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() == BASE_SHA
for path, expected in PINNED_FILES.items():
    content = subprocess.check_output(['git', 'show', 'origin/feature/unified-novelty:' + path], cwd=REPO)
    assert sha256(content).hexdigest() == expected, f'{path}: publish the exact verified source before running'
    target = REPO / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
print('Verified base commit and immutable runner/test file hashes')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU and restart'
PREPARED_DIR = Path('/kaggle/input/notebooks/dvdhere/preparation/review2_large_prepare_20261002T223303_224539Z')
TASK1_RUN_DIR = Path('/kaggle/input/notebooks/dvdhere/full-run-task1/full_task1_20261003T074354_763068Z')
DISCRIMINATION_EVIDENCE = Path('/kaggle/input/task2-discrimination-results/results.zip')  # Or complete extracted folder
OUTPUT_DIR = Path('/kaggle/working/task2_xgboost_diagnostic')
assert PREPARED_DIR.is_dir() and TASK1_RUN_DIR.is_dir(), 'Attach both complete verified input folders'
assert DISCRIMINATION_EVIDENCE.exists(), 'Attach original results.zip or set this to its extracted study folder'
assert not OUTPUT_DIR.exists(), 'Use a fresh output directory; never overwrite a prior test result'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Synthetic protocol tests plus read-only real-evidence verification; no research training here.
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_task2_xgboost_baseline.py',
                'tests/test_task2_discrimination.py'], cwd=REPO, check=True)
sys.path.insert(0, str(REPO))
from scripts.run_task2_xgboost_baseline import read_evidence
evidence, _ = read_evidence(DISCRIMINATION_EVIDENCE)
print('Verified replay rows:', len(evidence['replay_ids']))


In [ ]:
command = [sys.executable, 'scripts/run_task2_xgboost_baseline.py',
           '--prepared-dir', str(PREPARED_DIR), '--task1-run-dir', str(TASK1_RUN_DIR),
           '--discrimination-evidence', str(DISCRIMINATION_EVIDENCE), '--output-dir', str(OUTPUT_DIR)]
print('One locked GPU baseline:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)
print('Read protocol.json, history.json, training_manifest.json, validation_comparison.json, selection_lock.json, test_metrics.json and findings.json. Keep the full output private.')
